In [ ]:
# les notebooks vivent dans notebooks/ : on se replace à la racine PythonIPM
# pour que les chemins relatifs (EHCVM/, sorties/) et les imports du pipeline marchent
import os, sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RACINE)
sys.path.insert(0, str(RACINE / "pipeline"))


# Prise en main des bases EHCVM 2021 (Stata)

Trois étapes :

1. lire les fichiers `.dta` du dossier `EHCVM` ;
2. afficher les colonnes de chaque base et leur nom explicite dans le dictionnaire ;
3. donner la structure de chaque base (nombre de lignes et de colonnes).

Les noms explicites viennent de `dictionnaire_ehcvm.py`.

In [63]:
from pathlib import Path

import pandas as pd

import dictionnaire_ehcvm as dico

path = Path("EHCVM")

fichiers = sorted(f.name for f in path.glob("*.dta"))
print(len(fichiers), "bases dans le dossier :")
for f in fichiers:
    print(" -", f)

21 bases dans le dossier :
 - Base_Filets_de_Securite.dta
 - Base_Gouvernance.dta
 - Base_Individus.dta
 - Base_Menage.dta
 - Base_avoirs_du_menage.dta
 - Base_choc_COVID19.dta
 - Base_chocs.dta
 - Base_consommation_alimentaire.dta
 - Base_couts_Intrants_Agricoles.dta
 - Base_cultures.dta
 - Base_depenses_fetes.dta
 - Base_depenses_non_alimentaires.dta
 - Base_elevage.dta
 - Base_entreprises.dta
 - Base_equipement_agricole.dta
 - Base_gouvernance_(discrimination).dta
 - Base_parcelles_agricoles.dta
 - Base_peche.dta
 - Base_production.dta
 - Base_securite_alimentaire.dta
 - Base_transferts.dta


## 1. Lire les fichiers Stata

Les 4 bases utiles à l'IPM sont chargées en mémoire, dans un dictionnaire `bases`.

`convert_categoricals=False` garde les **codes** (1, 2, 3...) plutôt que les libellés :
c'est ce qu'il faut pour écrire des conditions du type `type_sanitaire == 1`.

In [64]:
bases = {}

for fichier in dico.BASES:
    bases[fichier] = pd.read_stata(path / fichier, convert_categoricals=False)
    print(fichier, "chargée")

menages = bases["Base_Menage.dta"]
individus = bases["Base_Individus.dta"]
avoirs = bases["Base_avoirs_du_menage.dta"]
secu_alim = bases["Base_securite_alimentaire.dta"]

Base_Menage.dta chargée
Base_Individus.dta chargée
Base_avoirs_du_menage.dta chargée
Base_securite_alimentaire.dta chargée


## 2. Colonnes et correspondance avec le dictionnaire

Pour chaque base : le code de la variable, son libellé Stata (le texte de la question),
et le nom explicite quand la variable figure dans le dictionnaire.

In [49]:
def colonnes(fichier):
    """Tableau des colonnes d'une base : code, nom explicite, libellé Stata."""
    lecteur = pd.io.stata.StataReader(path / fichier)
    libelles = lecteur.variable_labels()
    noms = dico.BASES.get(fichier, {})

    return pd.DataFrame({
        "variable": list(libelles),
        "nom_explicite": [noms.get(v, "") for v in libelles],
        "libelle": list(libelles.values()),
    })


In [50]:
# exemple : les colonnes de la base ménage qui ont un nom explicite
col_menage = colonnes("Base_Menage.dta")
col_menage[col_menage.nom_explicite != ""]

,variable,nom_explicite,libelle
2,hhid,id_menage,Idenfiant menage
3,grappe,grappe,Numero grappe
4,menage,menage,Numero menage
5,vague,vague,Vague
7,region,region,Region residence
8,milieu,milieu,Milieu residence
9,hhweight,ponderation_menage,Ponderation menage
10,hhsize,taille_menage,Taille menage
31,dtot,conso_totale_annuelle,Conso annuelle totale menage
32,pcexp,depense_par_tete,Indicateur de bien-être


In [51]:
col_individus = colonnes("Base_Individus.dta")
col_individus[col_individus.nom_explicite != ""]

,variable,nom_explicite,libelle
0,grappe,grappe,grappe
1,menage,menage,Identifiant du ménage
2,vague,vague,Vague
3,s01q00a,id_membre,membres__id
4,s01q01,sexe,1.01. Quel est le sexe de [NOM]
5,s01q02,lien_parente_cm,1.02. Quel est le lien de parenté de [NOM] ave...
7,s01q03b,mois_naissance,1.03b. Mois de naissance
8,s01q03c,annee_naissance,1.03c. Année de naissance
9,s01q04a,age_declare,1.04a. Quel âge avait [NOM] à son dernier anni...
11,s01q05,acte_naissance,1.05. Est-ce que [NOM] dispose d'un acte de na...


In [52]:
# les deux petites bases : toutes leurs colonnes tiennent à l'écran
print("=== Base_avoirs_du_menage.dta ===")
display(colonnes("Base_avoirs_du_menage.dta"))

print("=== Base_securite_alimentaire.dta ===")
display(colonnes("Base_securite_alimentaire.dta"))

=== Base_avoirs_du_menage.dta ===


,variable,nom_explicite,libelle
0,grappe,grappe,grappe
1,menage,menage,Identifiant du ménage
2,vague,vague,Vague
3,s12q01,code_bien,12.01.CODE D'ARTICLE
4,s12q00,,12.00. Qui est le répondant principal de la se...
5,s12q02,possede_bien,12.02. Un mbre du MEN possède [EQUIPEMENT] sva...
6,s12q03,nombre_biens,12.03. Quel est le nombre de [ARTICLE]?
7,s12q04,,12.04. Est ce que le bien appartient à un memb...
8,s12q05__0,,12.05. Quel est le CODE ID de la personne1 qui...
9,s12q05__1,,12.05. Quel est le CODE ID de la personne2 qui...


=== Base_securite_alimentaire.dta ===


,variable,nom_explicite,libelle
0,grappe,grappe,grappe
1,menage,menage,Identifiant du ménage
2,vague,vague,Vague
3,s08aq00,,8A.00. Code ID du principal répondant à la se...
4,s08aq01,fies_inquietude,"8A.01. 12 drnrs mois, inquiets de ne pas avoir..."
5,s08aq02,fies_pas_sain,"8A.02. 12 drnrs mois, pas pu manger une nourri..."
6,s08aq03,fies_peu_varie,"8A.03. 12 drnrs mois, avez-vous mangé une nour..."
7,s08aq04,fies_saute_repas,"8A.04. 12 drnrs mois, sauter un repas pas asse..."
8,s08aq05,fies_mange_moins,"8A.05. 12 drnrs mois, mangé moins que ce que v..."
9,s08aq06,fies_plus_de_nourriture,"8A.06. 12 drnrs mois, ménage n'avait plus de n..."


In [53]:
# combien de variables sont documentées, base par base ?
for fichier in dico.BASES:
    c = colonnes(fichier)
    print(f"{fichier:35s} {len(c):4d} variables, {(c.nom_explicite != '').sum():3d} nommées")

Base_Menage.dta                      309 variables,  38 nommées
Base_Individus.dta                   406 variables,  44 nommées
Base_avoirs_du_menage.dta             15 variables,   7 nommées
Base_securite_alimentaire.dta         14 variables,  13 nommées


## 3. Structure des bases

D'abord les 4 bases chargées, puis l'ensemble des bases du dossier — pour ces dernières on lit
seulement l'en-tête du fichier, sans charger les données (certaines pèsent 500 Mo).

In [54]:
for fichier, base in bases.items():
    lignes, colonnes_n = base.shape
    print(f"{fichier:35s} {lignes:7d} lignes x {colonnes_n:4d} colonnes")

Base_Menage.dta                       12965 lignes x  309 colonnes
Base_Individus.dta                    64491 lignes x  406 colonnes
Base_avoirs_du_menage.dta            583425 lignes x   15 colonnes
Base_securite_alimentaire.dta         13693 lignes x   14 colonnes


## 4. Niveau d'observation

Savoir ce que représente **une ligne** dans chaque base est le point le plus important avant
toute fusion : la clé du ménage est `grappe` + `menage` + `vague`.

In [56]:
cle = ["grappe", "menage", "vague"]

for fichier, base in bases.items():
    nb_menages = base[cle].drop_duplicates().shape[0]
    print(f"{fichier:35s} {len(base):7d} lignes pour {nb_menages:6d} ménages"
          f"  ({len(base) / nb_menages:.1f} ligne(s) par ménage)")

Base_Menage.dta                       12965 lignes pour  12965 ménages  (1.0 ligne(s) par ménage)
Base_Individus.dta                    64491 lignes pour  12965 ménages  (5.0 ligne(s) par ménage)
Base_avoirs_du_menage.dta            583425 lignes pour  12965 ménages  (45.0 ligne(s) par ménage)
Base_securite_alimentaire.dta         13693 lignes pour  12966 ménages  (1.1 ligne(s) par ménage)


Ce qu'on retient :

- **Base_Menage** : 1 ligne = 1 ménage (12 965) ;
- **Base_Individus** : 1 ligne = 1 membre du ménage (64 491, soit 5 personnes en moyenne) ;
- **Base_avoirs_du_menage** : 1 ligne = 1 bien proposé au ménage — exactement 45 lignes par
  ménage, y compris les biens qu'il ne possède pas (`possede_bien = 2`) ;
- **Base_securite_alimentaire** : 1 ligne = 1 ménage, mais la base contient **728 lignes
  entièrement vides** (clé `NaN`) — d'où les 13 693 lignes affichées au lieu de 12 965. Elles
  sont à supprimer avant toute fusion (`dropna(subset=cle)`).

Les 4 bases couvrent exactement les mêmes 12 965 ménages, ce qui permet de les fusionner sur
`grappe` + `menage` + `vague` sans perte.

## 5. Où se trouvent les indicateurs de l'IPM ?

Pour chaque indicateur de la proposition nationale : la base qui le porte et les variables à
utiliser. La colonne `verifiee` est calculée — elle confirme que les variables citées existent
bien dans la base et dans le dictionnaire.

In [ ]:
MENAGE_DTA = "Base_Menage.dta"
INDIVIDUS_DTA = "Base_Individus.dta"
AVOIRS_DTA = "Base_avoirs_du_menage.dta"
FIES_DTA = "Base_securite_alimentaire.dta"

indicateurs = [
    # dimension, indicateur, base, variables
    ("Education", "Fréquentation scolaire 6-16 ans", INDIVIDUS_DTA,
     ["scolarise_2021_2022", "scolarise_2020_2021", "a_frequente_ecole"]),
    ("Education", "10 années d'études, 17 ans et +", INDIVIDUS_DTA,
     ["diplome_plus_eleve", "niveau_instruction", "derniere_classe"]),
    ("Education", "Alphabétisation 17-49 ans", INDIVIDUS_DTA,
     ["lit_francais", "ecrit_francais"]),

    ("Sante", "Mortalité juvénile", "", []),                       # absent de l'EHCVM
    ("Sante", "Option : pas d'assurance maladie", INDIVIDUS_DTA, ["assurance_maladie"]),
    ("Sante", "Option : insécurité alimentaire (FIES)", FIES_DTA,
     ["fies_saute_repas", "fies_mange_moins", "fies_faim", "fies_journee_sans_manger"]),

    ("Emploi", "Chômage au sens du BIT", INDIVIDUS_DTA,
     ["a_travaille_7j", "emploi_mais_absent_7j", "recherche_emploi_30j_a",
      "recherche_emploi_30j_b", "disponible_emploi", "delai_disponibilite"]),

    ("Identification", "Acte de naissance", INDIVIDUS_DTA, ["acte_naissance"]),

    ("Conditions de vie", "Electricité", MENAGE_DTA, ["source_eclairage"]),
    ("Conditions de vie", "Logement (toit, murs, sol)", MENAGE_DTA,
     ["materiau_toit", "materiau_mur", "materiau_sol"]),
    ("Conditions de vie", "Eau potable", MENAGE_DTA, ["source_eau_boisson_seche"]),
    ("Conditions de vie", "Energie de cuisson", MENAGE_DTA,
     ["combustible_gaz", "combustible_electricite"]),
    ("Conditions de vie", "Toilettes privées améliorées", MENAGE_DTA,
     ["type_sanitaire", "sanitaire_partage"]),
    ("Conditions de vie", "Equipement du ménage", AVOIRS_DTA,
     ["code_bien", "possede_bien"]),

    ("Pondération", "Poids, taille, région, milieu", MENAGE_DTA,
     ["ponderation_menage", "taille_menage", "region", "milieu"]),
]


def variables_presentes(base, variables):
    """Toutes les variables citées existent-elles dans le dictionnaire de cette base ?"""
    if not base:
        return False
    connues = set(dico.BASES[base].values())
    return all(v in connues for v in variables)


carte = pd.DataFrame(indicateurs, columns=["dimension", "indicateur", "base", "variables"])
carte["disponible"] = carte.base != ""
carte["verifiee"] = [variables_presentes(b, v) for b, v in zip(carte.base, carte.variables)]
carte["variables"] = carte.variables.str.join(", ")

carte

In [ ]:
# quelles bases faut-il finalement retenir ?
bases_utiles = sorted(set(carte.loc[carte.disponible, "base"]))
print("bases nécessaires au calcul de l'IPM :")
for b in bases_utiles:
    lignes = len(bases[b])
    quoi = carte.loc[carte.base == b, "indicateur"].tolist()
    print(f"\n {b}  ({lignes} lignes)")
    for q in quoi:
        print("    -", q)

print("\nindicateur sans source dans l'EHCVM :",
      carte.loc[~carte.disponible, "indicateur"].tolist())

bases nécessaires au calcul de l'IPM :

 Base_Individus.dta  (64491 lignes)
    - Fréquentation scolaire 6-16 ans
    - 10 années d'études, 17 ans et +
    - Alphabétisation 17-49 ans
    - Option : pas d'assurance maladie
    - Chômage au sens du BIT
    - Acte de naissance

 Base_Menage.dta  (12965 lignes)
    - Electricité
    - Logement (toit, murs, sol)
    - Eau potable
    - Energie de cuisson
    - Toilettes privées améliorées
    - Poids, taille, région, milieu

 Base_avoirs_du_menage.dta  (583425 lignes)
    - Equipement du ménage

 Base_securite_alimentaire.dta  (13693 lignes)
    - Option : insécurité alimentaire (FIES)

indicateur sans source dans l'EHCVM : ['Mortalité juvénile']


Bilan : **4 bases suffisent** pour l'IPM — Individus (éducation, emploi, identification, santé),
Ménage (conditions de vie et pondérations), Avoirs (équipement) et Sécurité alimentaire (option
santé). Les 18 autres bases du dossier (consommation, agriculture, chocs, transferts...) ne
servent pas au calcul.

Seule exception : la **mortalité juvénile** n'existe nulle part dans l'EHCVM. `Base_chocs`
contient bien « Décès d'un membre du ménage » (code 102, 1 328 ménages sur 3 ans, avec la date),
mais **sans l'âge du défunt** — donc impossible d'isoler les moins de 18 ans.

## 6. Renommer les colonnes avec le dictionnaire

Maintenant qu'on sait quelles bases servent, on remplace les codes de variables (`s11q54`,
`s02q33`...) par les noms explicites du dictionnaire, directement dans les bases chargées.

Les variables absentes du dictionnaire gardent leur code : le renommage ne supprime ni ne
modifie aucune colonne.

In [60]:
# renommage de chaque base avec le dictionnaire correspondant
for fichier, noms in dico.BASES.items():
    bases[fichier] = bases[fichier].rename(columns=noms)
    renommees = [c for c in bases[fichier].columns if c in noms.values()]
    print(f"{fichier:35s} {len(renommees):3d} colonnes renommées "
          f"sur {bases[fichier].shape[1]}")

# on met à jour les raccourcis
menages = bases["Base_Menage.dta"]
individus = bases["Base_Individus.dta"]
avoirs = bases["Base_avoirs_du_menage.dta"]
secu_alim = bases["Base_securite_alimentaire.dta"]

# vérification : les noms explicites sont bien en place
menages[["id_menage", "ponderation_menage", "taille_menage", "region", "milieu",
         "type_sanitaire", "source_eclairage", "materiau_toit"]].head()

Base_Menage.dta                      38 colonnes renommées sur 309
Base_Individus.dta                   44 colonnes renommées sur 406
Base_avoirs_du_menage.dta             7 colonnes renommées sur 15
Base_securite_alimentaire.dta        13 colonnes renommées sur 14


,id_menage,ponderation_menage,taille_menage,region,milieu,type_sanitaire,source_eclairage,materiau_toit
0,102.0,1098.117188,7,1,1,1,1,3
1,103.0,1098.117188,3,1,1,1,1,3
2,104.0,1098.117188,1,1,1,1,1,3
3,105.0,1098.117188,1,1,1,2,1,3
4,106.0,1098.117188,5,1,1,6,1,3


In [62]:
menages.head()

,country,year,id_menage,grappe,menage,vague,zae,region,milieu,ponderation_menage,...,s20cq25b__1,s20cq25b__2,s20cq25b__3,s20cq25b__4,s20cq25b__5,s20cq26,s20cq27a,s20cq27b,s20cq27b_autre,s20cq28
0,CIV,2021,102.0,1,2.0,1.0,6,1,1,1098.117188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
1,CIV,2021,103.0,1,3.0,1.0,6,1,1,1098.117188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
2,CIV,2021,104.0,1,4.0,1.0,6,1,1,1098.117188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
3,CIV,2021,105.0,1,5.0,1.0,6,1,1,1098.117188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
4,CIV,2021,106.0,1,6.0,1.0,6,1,1,1098.117188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN


## 7. La matrice situationnelle X

Méthode Alkire-Foster, telle que présentée au chapitre 4 du guide *L'Indice de pauvreté
multidimensionnelle au service des ODD* :

| Étape | Objet | Contenu |
|---|---|---|
| 1 | **matrice situationnelle X** | la *situation* de chaque unité dans chaque indicateur (années de scolarité, accès à l'assainissement...) |
| 2 | vecteur de seuils **z** | le seuil de privation de chaque indicateur |
| 3 | matrice de privation **g⁰** | 1 si la situation est en deçà du seuil, 0 sinon |
| 4 | pondérations **w** → score **cᵢ** | somme pondérée des privations |
| 5 | seuil **k** = 1/3 → matrice censurée **g⁰(k)** | H, A puis M₀ = H × A |

Ici on construit **uniquement X**. Aucun seuil n'est appliqué : les colonnes contiennent des
situations brutes (des effectifs, un maximum, un code de modalité), donc rien n'est encore décidé
sur les privations.

**Une ligne = un ménage** (12 965), conformément au document méthodologique de l'IPM-CI : le score
de privation se calcule au niveau du ménage, chaque ménage comptant ensuite nᵢ fois dans l'indice.

Pour les indicateurs mesurés sur les personnes, X contient **deux colonnes** : l'effectif concerné
(qui est éligible) et l'effectif en situation défavorable. C'est ce qui permettra plus tard de
distinguer « non privé » de « non concerné » sans revenir aux données individuelles.

In [61]:
import numpy as np

# les bases portent déjà les noms explicites (section 6)
ind = individus.copy()
men = menages.copy()
avo = avoirs.copy()
fies = secu_alim.dropna(subset=cle)   # 728 lignes vides dans la base FIES

# --- âge : année d'enquête (vague 1 = 2021, vague 2 = 2022) moins année de naissance ---
ind["age"] = np.where(ind.vague == 1, 2021, 2022) - ind.annee_naissance
ind["age"] = ind.age_declare.fillna(ind.age)

# --- années d'études = années accomplies avant le niveau + classe atteinte ---
ANNEES_AVANT_NIVEAU = {
    1: 0,   # maternelle
    2: 0,   # primaire        (1re à 6e année)
    3: 6,   # secondaire 1    (6e a 3e  -> 7e a 10e annee)
    4: 6,   # secondaire 1 technique
    5: 10,  # secondaire 2    (2nde a Tle)
    6: 10,  # secondaire 2 technique
    7: 13,  # post-secondaire
    8: 13,  # supérieur
}
etudes_achevees = ind.niveau_instruction.map(ANNEES_AVANT_NIVEAU) + ind.derniere_classe
etudes_en_cours = ind.niveau_en_cours.map(ANNEES_AVANT_NIVEAU) + ind.classe_en_cours - 1
ind["annees_etudes"] = etudes_achevees.fillna(etudes_en_cours)
ind.loc[ind.a_frequente_ecole == 2, "annees_etudes"] = 0   # n'a jamais fréquenté l'école

# --- autres situations individuelles ---
ind["scolarise"] = (ind.scolarise_2021_2022 == 1) | (ind.scolarise_2020_2021 == 1)
ind["alphabetise"] = (ind.lit_francais == 1) & (ind.ecrit_francais == 1)

sans_emploi = (ind.a_travaille_7j != 1) & (ind.emploi_mais_absent_7j != 1)
recherche = (ind.recherche_emploi_30j_a == 1) | (ind.recherche_emploi_30j_b == 1)
disponible = ind.delai_disponibilite.isin([1, 2, 3]) | (ind.disponible_emploi == 1)
ind["chomeur_bit"] = sans_emploi & recherche & disponible   # les 3 critères du BIT

ind[["age", "annees_etudes", "scolarise", "alphabetise", "chomeur_bit"]].describe()

AttributeError: 'DataFrame' object has no attribute 'niveau_en_cours'

In [ ]:
# --- 7.1 situations issues des individus, résumées par ménage -----------------
groupes = ind.groupby(cle)


def par_menage(fonction):
    """Applique une fonction à chaque ménage et renvoie une série indexée par la clé."""
    return groupes.apply(fonction, include_groups=False)


X_individus = pd.DataFrame({
    # éducation
    "enfants_6_16": par_menage(lambda d: d.age.between(6, 16).sum()),
    "enfants_6_16_non_scolarises": par_menage(
        lambda d: (d.age.between(6, 16) & ~d.scolarise).sum()),
    "membres_17_95": par_menage(lambda d: d.age.between(17, 95).sum()),
    "annees_etudes_max": par_menage(lambda d: d.annees_etudes[d.age.between(17, 95)].max()),
    "membres_17_49": par_menage(lambda d: d.age.between(17, 49).sum()),
    "membres_17_49_alphabetises": par_menage(
        lambda d: (d.age.between(17, 49) & d.alphabetise).sum()),
    # emploi
    "membres_16_35": par_menage(lambda d: d.age.between(16, 35).sum()),
    "chomeurs_16_35": par_menage(lambda d: (d.age.between(16, 35) & d.chomeur_bit).sum()),
    # identification
    "enfants_5_15": par_menage(lambda d: d.age.between(5, 15).sum()),
    "enfants_5_15_sans_acte": par_menage(
        lambda d: (d.age.between(5, 15) & (d.acte_naissance != 1)).sum()),
    # santé (option)
    "membres_assures": par_menage(lambda d: (d.assurance_maladie == 1).sum()),
})

X_individus.head()

In [ ]:
# --- 7.2 situations du ménage : combustible principal, biens, sécurité alimentaire ---

# le combustible de rang 1 parmi les 8 colonnes de la question 11.52
COMBUSTIBLES = ["combustible_bois_ramasse", "combustible_bois_achete", "combustible_charbon",
                "combustible_gaz", "combustible_electricite", "combustible_petrole",
                "combustible_dechets_animaux", "combustible_autre"]
rangs = men[COMBUSTIBLES]
men["combustible_principal"] = np.where(
    rangs.eq(1).any(axis=1),
    rangs.eq(1).idxmax(axis=1).str.replace("combustible_", ""),
    None)

# biens : 7 petits biens du MPI + la voiture, comptés depuis la base des avoirs
PETITS_BIENS = [16, 19, 20, 29, 30, 35, 37]   # frigo, radio, TV, moto, vélo, portable, PC
VOITURE = 28
avo["possede"] = avo.possede_bien == 1
groupes_avoirs = avo.groupby(cle)
X_biens = pd.DataFrame({
    "nb_petits_biens": groupes_avoirs.apply(
        lambda d: d.possede[d.code_bien.isin(PETITS_BIENS)].sum(), include_groups=False),
    "possede_voiture": groupes_avoirs.apply(
        lambda d: int(d.possede[d.code_bien == VOITURE].any()), include_groups=False),
})

# score FIES : nombre de « oui » sur les 8 questions (98 NSP et 99 Refus = manquant)
ITEMS_FIES = ["fies_inquietude", "fies_pas_sain", "fies_peu_varie", "fies_saute_repas",
              "fies_mange_moins", "fies_plus_de_nourriture", "fies_faim",
              "fies_journee_sans_manger"]
reponses = fies.set_index(cle)[ITEMS_FIES].replace({98: np.nan, 99: np.nan, 2: 0})
X_fies = pd.DataFrame({"score_fies": reponses.sum(axis=1, min_count=8)})

print(men.combustible_principal.value_counts(dropna=False).to_dict())
X_biens.join(X_fies).head()

In [ ]:
# --- 7.3 assemblage de la matrice situationnelle X ----------------------------
COLONNES_MENAGE = [
    # conditions de vie
    "source_eclairage", "materiau_toit", "materiau_mur", "materiau_sol",
    "source_eau_boisson_seche", "distance_source_eau_seche", "combustible_principal",
    "type_sanitaire", "sanitaire_partage",
    # identification du ménage, pondération et désagrégation
    "id_menage", "ponderation_menage", "taille_menage", "region", "milieu",
]

X = (men.set_index(cle)[COLONNES_MENAGE]
        .join(X_individus)
        .join(X_biens)
        .join(X_fies))

print("matrice situationnelle X :", X.shape[0], "ménages x", X.shape[1], "colonnes")
X.head()

In [ ]:
# --- 7.4 contrôles : que contient X ? -----------------------------------------
print("--- effectifs concernés (éligibilité) ---")
for c in ["enfants_6_16", "membres_17_95", "membres_17_49", "membres_16_35", "enfants_5_15"]:
    print(f"{c:20s} ménages sans personne concernée : {(X[c] == 0).sum():5d}"
          f"  ({(X[c] == 0).mean():.1%})")

print("\n--- situations chiffrées ---")
display(X[["annees_etudes_max", "enfants_6_16_non_scolarises", "chomeurs_16_35",
           "enfants_5_15_sans_acte", "nb_petits_biens", "score_fies",
           "taille_menage"]].describe().round(2))

X.to_stata("matrice_situationnelle_ehcvm2021.dta", write_index=True, version=118)
print("enregistré : matrice_situationnelle_ehcvm2021.dta")

print("\n--- valeurs manquantes ---")
X.isna().mean().sort_values(ascending=False).head(6).round(3)

### Lecture de X et étape suivante

X compte 12 965 lignes et 28 colonnes. Ce que les contrôles montrent déjà :

- **l'éligibilité n'est pas marginale** : 33 % des ménages n'ont aucun enfant de 6-16 ans, 32 %
  aucun enfant de 5-15 ans, 25 % aucun membre de 16-35 ans. Pour ces ménages la question n'est pas
  « sont-ils privés ? » mais « sont-ils concernés ? » — d'où les colonnes d'effectifs ;
- `annees_etudes_max` : médiane 5 années, 3ᵉ quartile 10 années — le seuil de 10 années de la
  proposition nationale tombe donc autour du quart supérieur des ménages ;
- `sanitaire_partage` est vide pour 24,6 % des ménages : la question 11.55 n'est pas posée à ceux
  qui n'ont **aucune toilette**. Ce vide vaut privation, pas information manquante ;
- `score_fies` manque pour 1,5 % des ménages, `annees_etudes_max` pour 6 ménages sans personne de
  17-95 ans.

Étape suivante du chapitre 4 : le **vecteur de seuils z**, puis la matrice de privation g⁰.